# Question 13 - ViT Image Patching (Fashion-MNIST, P = 7, D = 64)
1. **N = (28/7)^2 = 16 patches**
2. Single flattened patch: 7 x 7 x 1 = **49**, shape **(49,)**
3. Sequence into the encoder: 16 patches + 1 [class] token, giving **(17, 64)**, or (B, 17, 64) with a batch.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
(x_train, _), _ = keras.datasets.fashion_mnist.load_data()
x = (x_train[:4].astype("float32") / 255.0)[..., None]    # (4, 28, 28, 1)
P, D = 7, 64

patches = tf.image.extract_patches(x, sizes=[1, P, P, 1], strides=[1, P, P, 1], rates=[1, 1, 1, 1], padding="VALID")
patches = tf.reshape(patches, (x.shape[0], -1, P * P))
print("N =", patches.shape[1], "| single flattened patch shape:", patches[0, 0].shape)

In [ ]:
class ViTEmbedding(layers.Layer):
    def __init__(self, num_patches=16, dim=64):
        super().__init__()
        self.proj = layers.Dense(dim)
        self.cls = self.add_weight(shape=(1, 1, dim), initializer="zeros", name="cls")
        self.pos = layers.Embedding(num_patches + 1, dim)

    def call(self, patches):
        b = tf.shape(patches)[0]
        tokens = self.proj(patches)
        tokens = tf.concat([tf.tile(self.cls, [b, 1, 1]), tokens], axis=1)
        return tokens + self.pos(tf.range(tf.shape(tokens)[1]))

seq = ViTEmbedding()(patches)
print("sequence fed to encoder:", seq.shape)

In [ ]:
plt.figure(figsize=(4, 4))
for i in range(16):
    plt.subplot(4, 4, i + 1); plt.imshow(patches[0, i].numpy().reshape(7, 7), cmap='gray'); plt.axis('off')
plt.suptitle("16 patches of one image"); plt.show()